## Functions

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import sys, os
from ipywidgets import interact
from IPython.display import display, Math
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Math
from scipy.special import genlaguerre, legendre, lpmv
from matplotlib import cm
from scipy.special import sph_harm, genlaguerre, factorial
import matplotlib as mpl
%matplotlib inline

In [4]:
def xsph(theta, phi, r):
    return np.cos(phi)*np.sin(theta)*r

def ysph(theta, phi, r):
    return np.sin(phi)*np.sin(theta)*r

def zsph(theta, phi, r):
    return np.cos(theta)*r


def az(phi,m):
    return np.exp(1j*m*phi)
def polar(theta, l, m):
    return lpmv(abs(m), l, np.cos(theta))

def ang_norm(l,m):
    num = (2*l+1)*factorial(l-abs(m))
    den = 4*np.pi*factorial(l+abs(m))
    return (num/den)**.5

def Ylm(theta, phi, l, m):
    sign = 1
    if not (m==0):
        sign = (-m/abs(m))**abs(m)
    return ang_norm(l,m)*az(phi,m)*polar(theta,l,m)*sign

a0 = 5.29e-11 #m
def norm(n,l):
    return (factorial(n-l-1)/(2*n*factorial(n+l)))**.5 * (2/(n*a0))**(l+1.5)

def Rnl(r, n, l):
    gl = genlaguerre(n-l-1,2*l+1)(2*r/(n*a0))
    p1 = r**l * np.exp(-r/(n*a0))
    return norm(n,l)*gl*p1
cmap = LinearSegmentedColormap.from_list(
    "bright_div",
    [[1,0,0], "black", [0,.3,1]]
)

In [5]:
def sph_harm():
    lwidget = widgets.IntSlider(min=0, max=4, value=1)
    mwidget = widgets.IntSlider(min=-4, max=4, value=0)
    @interact(
    elevation = widgets.FloatSlider(min=-90, max=90, step=.1,value=30),
    azim = widgets.FloatSlider(min=0, max=180, step=.1,value=30),
    l = lwidget,
    m =mwidget,
    part = widgets.RadioButtons(options=['Real','Imaginary', 'Probability'],value='Real',description=' ')
    )
    def g(**kargs):
        m = kargs['m']
        l = kargs['l']

        if m > l:
            mwidget.value=l
            m=l
        if m < -l:
            mwidget.value=-l
            m=-l
        theta = np.linspace(0, np.pi, 100)
        phi = np.linspace(0, 2*np.pi, 200)

        Theta, Phi = np.meshgrid(theta, phi)
        X = xsph(Theta,Phi,1)
        Y = ysph(Theta,Phi,1)
        Z = zsph(Theta,Phi,1)

        Phi = np.atan2(Y,X)
        r = np.sqrt(X**2+Y**2+Z**2)
        Theta = np.acos(Z/r)

        if kargs['part']=='Real':
            psi = np.real(Ylm(Theta,Phi,l,m))
            norm = plt.Normalize(-psi.max(),psi.max())
            color = cmap(norm(psi))
            clabel='Re($\\psi$)'
        elif kargs['part']=='Imaginary':
            psi = np.imag(Ylm(Theta,Phi,l,m))
            if not (m==0):
                norm = plt.Normalize(-psi.max(),psi.max())
            else:
                norm = plt.Normalize(-1,1)
            color = cmap(norm(psi))
            clabel='Im($\\psi$)'

        elif kargs['part']=='Probability':
            psi = Ylm(Theta,Phi,l,m)
            psi = np.real(psi * np.conjugate(psi))
            norm = plt.Normalize(0,psi.max())
            color = cm.grey_r(norm(psi))
            clabel='($\\psi^*\\psi$)'
        
        #fig.colorbar(mappable, ax=ax, shrink=0.6)
        fig = plt.figure()
        gs = GridSpec(1,1,fig)
        ax = fig.add_subplot(gs[0],projection='3d')
        ax.plot_surface(X,Y,Z, facecolors=color, shade=False)
        ax.set_aspect('equal')
        ax.view_init(elev=kargs['elevation'], azim=kargs['azim'])
        
        mappable = mpl.cm.ScalarMappable(norm=norm, cmap=cmap if kargs['part']!='Probability' else cm.grey_r)
        mappable.set_array([])  # required for colorbar
        fig.colorbar(mappable, ax=ax, shrink=0.6,label=clabel)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title(f'$Y_{l}^{{{m}}}$')
        ax.set_axis_off()
        plt.show()

sph_harm()

interactive(children=(FloatSlider(value=30.0, description='elevation', max=90.0, min=-90.0), FloatSlider(value…

In [8]:
def radial():
    lwidget = widgets.IntSlider(min=0, max=5, step=1, value=0)
    savewidget = widgets.ToggleButton(value=False, description='Save')
    @interact(
    n = widgets.IntSlider(min=1, max=5, step=1, value=2),
    l = lwidget,
    prob = widgets.ToggleButton(),
    filename = widgets.Text(value='R.svg'),
    save= savewidget
    )
    def g(**kargs):
        if kargs['l'] > kargs['n']-1:
            lwidget.value=kargs['n']-1
            l = kargs['n']-1
        else:
            l=kargs['l']
        r = np.linspace(0, 40*a0, 200)
        fig = plt.figure(figsize=(4,3))
        gs= GridSpec(1,1,fig)
        axs = []
        axs.append(fig.add_subplot(gs[0]))

        ax = axs[0]
        r = np.linspace(0, 30*a0, 200)
        if (kargs['prob']):
            ax.plot(r*1e10, Rnl(r, kargs['n'], l)**2*r**2)
            ax.axhline(0, color='k', linestyle='--')
            ax.set_xlim(0, r[-1]*1e10)
            ax.set_xlabel('r($\\AA$)')
            ax.set_ylabel('$r^2 R(r)^2$ (m$^{-3}$)')
            #ax.set_ylim(0,1.2e10)
        if (not kargs['prob']):
            ax.plot(r*1e10, Rnl(r, kargs['n'], l))
            ax.axhline(0, color='k', linestyle='--')
            ax.set_xlim(0, r[-1]*1e10)
            ax.set_xlabel('r($\\AA$)')
            ax.set_ylabel('$R(r)$ (m$^{-3/2}$)')
            #ax.set_ylim(-1e15,5e15)

        if kargs['save']:
            plt.savefig(kargs['filename'])
            savewidget.value=False
        #ax.set_ylim(0,1e10)

        plt.show()
radial()

interactive(children=(IntSlider(value=2, description='n', max=5, min=1), IntSlider(value=0, description='l', m…